# Personalized Career Recommendation System — Model Evaluation

This notebook evaluates the recommendation engine used in this project.

## What we built
| Item | Details |
|------|--------|
| **Approach** | Content-based career recommendation |
| **Skill matching** | TF–IDF (manual formulas in `formulas.py`) |
| **Interest matching** | Token overlap score |
| **Similarity** | Cosine similarity on skill+interest token vectors |
| **Final score** | `(TF-IDF × 0.55) + (Interest × 0.27) + (Cosine×100 × 0.18)` |
| **Output** | Top-3 ranked careers + match % |
| **Dataset** | `personalized/data/career_dataset.csv` (~15,100 rows, 151 IT careers) |
| **Engine** | `CareerAdvisorEngine` in `personalized/career.py` |

## Metrics reported
- **Top-1 Accuracy** — exact #1 career correct
- **Top-3 Accuracy** — correct career appears in top 3 (primary for this app)
- **Precision / Recall / F1** — macro-averaged over careers


## 1. Imports

Uses this repo’s `evaluate_model.py`, which calls `CareerAdvisorEngine` and `formulas.py` (same scoring as production).

In [ ]:
import os
import sys
import pandas as pd

# Run from repo root or personalized/
ROOT = os.path.abspath(os.path.join(os.getcwd()))
PERSONALIZED = ROOT if ROOT.endswith("personalized") else os.path.join(ROOT, "personalized")
if not os.path.isdir(PERSONALIZED):
    PERSONALIZED = os.path.join(os.path.dirname(ROOT), "personalized")

sys.path.insert(0, PERSONALIZED)
os.chdir(PERSONALIZED)

from evaluate_model import evaluate_model, DEFAULT_CSV
from formulas import WEIGHT_SKILL, WEIGHT_INTEREST, WEIGHT_COSINE

print("Working dir:", os.getcwd())
print("Dataset:", DEFAULT_CSV)
print("Weights:", WEIGHT_SKILL, WEIGHT_INTEREST, WEIGHT_COSINE)

## 2. Dataset overview

In [ ]:
df = pd.read_csv(DEFAULT_CSV)
df.columns = df.columns.str.strip()

print("Rows   :", len(df))
print("Columns:", list(df.columns))
print("Careers:", df["Recommended_Career"].nunique())
print("\nSample careers:")
print(sorted(df["Recommended_Career"].astype(str).unique())[:20], "...")
df.head(3)

## 3. Run evaluation (80/20 × 5 epochs)

This can take **~10–15 minutes** on the full 15k-row dataset (cosine over a large vocabulary).

For a quicker check during development, set `EPOCHS = 1`.

In [ ]:
EPOCHS = 5  # use 1 for a fast smoke test

results = evaluate_model(csv_path=DEFAULT_CSV, epochs=EPOCHS)

epoch_df = pd.DataFrame(results["epoch_results"])
summary = results["summary"]

display(epoch_df)
pd.DataFrame([summary])

## 4. Overall model accuracy (summary table)

Last measured full run on this project’s current dataset (`personalized/data/career_dataset.csv`):

In [ ]:
# Reference results from full 5-epoch evaluation on current dataset
# (re-run Section 3 to refresh these numbers live)
reference_summary = {
    "Top-1 Accuracy (%)": 43.72,
    "Top-3 Accuracy (%)": 74.81,
    "Precision": 0.583,
    "Recall": 0.472,
    "Overall F1-Score": 0.521,
    "Dataset rows": 15100,
    "Career classes": 151,
    "Train/Test": "80% / 20%",
    "Epochs": 5,
    "Algorithm": "Content-based TF-IDF + Interest + Cosine (0.55/0.27/0.18)",
}

pd.DataFrame([reference_summary]).T.rename(columns={0: "Score"})

## 5. Run from terminal

```bash
cd personalized
python evaluate_model.py 5
```

Primary metric for this app is Top-3 accuracy because the UI returns three careers.
